# k-means clustering

k-means is **unsupervised**: given unlabeled points, it partitions them into $k$
groups by iteratively assigning each point to the nearest cluster centre. We use
[`linfa-clustering`](https://docs.rs/linfa-clustering).

Unlike the regression chapters, there are no target labels — just the feature
matrix (the `ndarray` from [Chapter 1](../01-foundations/ndarray-basics.ipynb)).

In [ ]:
:dep ndarray = { version = "0.15" }
:dep linfa = { version = "0.7" }
:dep linfa-clustering = { version = "0.7" }
use ndarray::array;

// Two visually separated blobs of points (no labels).
let data = array![
    [1.0_f64, 1.0], [1.2, 0.8], [0.8, 1.1],
    [5.0, 5.0], [5.2, 4.8], [4.9, 5.1]
];
println!("{} unlabeled points", data.nrows());

In [ ]:
use linfa::prelude::*;
use linfa::DatasetBase;
use linfa_clustering::KMeans;
use ndarray::Array1;

// Unsupervised data uses `DatasetBase::from(records)` — records only.
// The explicit `: Array1<usize>` lets evcxr persist the result across cells
// (a block-bound variable needs its type named — see the crate reference).
let assignments: Array1<usize> = {
    let dataset = DatasetBase::from(data.clone());
    let model = KMeans::params(2)
        .max_n_iterations(100)
        .fit(&dataset)
        .expect("KMeans fit failed");
    model.predict(&dataset)
};
println!("cluster assignments: {:?}", assignments);

The three low points landed in one cluster and the three high points in the
other. Let's colour the scatter plot by cluster:

In [ ]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use plotters::prelude::*;

evcxr_figure((420, 360), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("k-means clusters", ("sans-serif", 18))
        .margin(10)
        .x_label_area_size(30)
        .y_label_area_size(30)
        .build_cartesian_2d(0f64..6f64, 0f64..6f64)?;
    chart.configure_mesh().draw()?;
    chart.draw_series((0..data.nrows()).map(|i| {
        let colour = if assignments[i] == 0 { RED } else { BLUE };
        Circle::new((data[[i, 0]], data[[i, 1]]), 5, colour.filled())
    }))?;
    Ok(())
})

k-means needs you to choose $k$ up front and assumes roughly round clusters.
Next: [DBSCAN](dbscan.ipynb), which finds clusters of arbitrary shape and
discovers the number of clusters on its own.